<a href="https://colab.research.google.com/github/Abrar-404/AI-ML_Practices_and_Assignments/blob/main/Eid_Vacation_Practice_Advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing, load_diabetes, load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from ucimlrepo import fetch_ucirepo

In [3]:
pip install ucimlrepo

# Q1  The Income Trap

# DATASET Adult Income (Census Income)  
https://archive.ics.uci.edu/dataset/2/adult

### RESEARCH
### Read about what 'capital-gain' and 'capital-loss' represent in US income

## Theory
1. The 'capital-gain' column has ~91% of its values equal to exactly zero, with a
few extreme values reaching over $99,000. Is this an outlier problem or a
distributional problem? Justify your answer using the concept of the Normal
Distribution from class.
2. If you apply StandardScaler (Z-score normalization) to 'capital-gain', what
happens to the Z-scores of those extreme values? Why would a RobustScaler be
more appropriate here?  
3. Explain Winsorization. If you apply 5th–95th percentile Winsorization to
'hours-per-week', what are you assuming about the data-generating process, and
what are the risks?


##  Practical
4. Load the dataset (it uses '?' for missing values — handle this first). Perform
train-test split. Plot the distribution of 'capital-gain' using a KDE plot. Confirm
the extreme skew visually.
5. Apply Winsorization (clip at 5th–95th percentile) to 'capital-gain' on the training
set. Apply the SAME clipping bounds to the test set. Plot before/after KDE
plots.
6. Apply RobustScaler to 'hours-per-week' and 'capital-gain' . Compare the scaled
range with what StandardScaler would have produced. Which handles the
outliers better? Justify numerically.

#### THEORY

1. It's a distributional problem. A Normal Distribution assumes values spread symmetrically around a mean. Here, 91% are exactly 0 and a few go to $99k. This is a zero-inflated distribution, not a Normal one with outliers. The extreme values aren't errors — they're real wealthy people's capital gains. You can't "fix" this with outlier removal.

2. StandardScaler computes Z = (x - mean) / std. Since 91% of values are 0 and a few are huge, the mean gets pulled up and the std gets inflated by those extremes. The Z-scores of the extreme values won't look as extreme as they should, and the Z-scores of the zeros will become oddly negative. RobustScaler uses median and IQR instead — since the median of capital-gain is 0 (most people have 0), it's not affected by the extreme values at all.

3. Instead of removing outliers, you cap them. If you apply 5th–95th percentile Winsorization to hours-per-week:
- Any value below the 5th percentile gets replaced with the 5th percentile value
- Any value above the 95th percentile gets replaced with the 95th percentile value.

Assumption I'm making:
Values beyond those percentiles are measurement errors or noise, not real data points.

Risks:

If those extreme values are real (someone genuinely works 90 hrs/week), it's distorting the truth
It must be computed the bounds on the training set only, then apply the same bounds to the test set — otherwise it's leaking test data information

In [7]:
# load dataset
adult = fetch_ucirepo(id=2)
X = adult.data.features
y = adult.data.targets

df_adult = pd.concat([X, y], axis = 1)

df_adult

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States,<=50K.
48838,64,NaN,321403,HS-grad,9,Widowed,NaN,Other-relative,Black,Male,0,0,40,United-States,<=50K.
48839,38,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States,<=50K.
48840,44,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States,<=50K.
